## Day 2: Classification in Python



## Learning Outcomes
- Logistic regression
- Sigmoid functions
- Log loss and regularisation
- Classification thresholds
- Confusion matrices
- Accuracy, Recall, Precision, F1
- ROC and AUC
- Multiclass classification
- Feature selection
- Decision trees and random forests
- Cross-validation
- Overfitting and underfitting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import *
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import permutation_importance


## Example 1: Breast Cancer Dataset

In [ ]:
cancer=load_breast_cancer(as_frame=True)
df=cancer.frame
df.head()

### Logistic Regression

In [ ]:
X=df.drop(columns="target")
y=df.target
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)
model=LogisticRegression(max_iter=5000)
model.fit(X_train,y_train)
probs=model.predict_proba(X_test)[:,1]
preds=model.predict(X_test)
print("Accuracy",accuracy_score(y_test,preds))

## Sigmoid Function

In [ ]:
x=np.linspace(-10,10,200)
y=1/(1+np.exp(-x))
plt.plot(x,y)
plt.title("Sigmoid Function")
plt.show()

## Classification Thresholds

In [ ]:
for t in [0.2,0.5,0.8]:
    pred=(probs>t).astype(int)
    print(t,accuracy_score(y_test,pred))

## Confusion Matrix

In [ ]:
cm=confusion_matrix(y_test,preds)
sns.heatmap(cm,annot=True,fmt="d")
plt.show()

## Accuracy, Recall and Precision

In [ ]:
print("Accuracy",accuracy_score(y_test,preds))
print("Recall",recall_score(y_test,preds))
print("Precision",precision_score(y_test,preds))
print("F1",f1_score(y_test,preds))

## ROC Curve and AUC

In [ ]:
fpr,tpr,_=roc_curve(y_test,probs)
plt.plot(fpr,tpr)
plt.plot([0,1],[0,1],"--")
plt.show()
print("AUC",roc_auc_score(y_test,probs))

## Regularisation

In [ ]:
for c in [0.01,0.1,1,10]:
    m=LogisticRegression(C=c,max_iter=5000)
    m.fit(X_train,y_train)
    print(c,m.score(X_test,y_test))

## Example 2: Multiclass Classification with Iris

In [ ]:
iris=load_iris(as_frame=True)
X=iris.data
y=iris.target
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)
clf=LogisticRegression(max_iter=5000,multi_class="auto")
clf.fit(X_train,y_train)
print(classification_report(y_test,clf.predict(X_test)))

## Feature Selection

In [ ]:
selector=SelectKBest(mutual_info_classif,k=5)
selector.fit(X_train,y_train)
print(X.columns[selector.get_support()])

## Decision Trees

In [ ]:
tree=DecisionTreeClassifier(max_depth=3,random_state=42)
tree.fit(X_train,y_train)
plt.figure(figsize=(10,5))
plot_tree(tree,filled=True)
plt.show()

## Random Forests

In [ ]:
rf=RandomForestClassifier(n_estimators=200,random_state=42)
rf.fit(X_train,y_train)
print("Accuracy",rf.score(X_test,y_test))

## Variable Importance

In [ ]:
imp=permutation_importance(rf,X_test,y_test,n_repeats=10,random_state=42)
pd.Series(imp.importances_mean,index=X.columns).sort_values().plot.barh()
plt.show()

## Validation Set Approach

In [ ]:
scores=cross_val_score(rf,X,y,cv=5)
print(scores)
print(scores.mean())

## LOOCV and K-fold CV

In [ ]:
cv=KFold(n_splits=10,shuffle=True,random_state=42)
print(cross_val_score(rf,X,y,cv=cv).mean())

## Overfitting Example

In [ ]:
X,y=make_classification(n_samples=500,n_features=20,n_informative=3,random_state=42)
Xtr,Xte,ytr,yte=train_test_split(X,y,random_state=42)
for d in [1,3,10,None]:
    m=DecisionTreeClassifier(max_depth=d,random_state=42)
    m.fit(Xtr,ytr)
    print(d,m.score(Xtr,ytr),m.score(Xte,yte))

## Activities
1. Change classification thresholds and compare recall and precision.
2. Compare logistic regression, decision trees and random forests.
3. Repeat feature selection with different numbers of predictors.
4. Investigate overfitting by increasing tree depth.
5. Compare 5-fold and 10-fold cross-validation.